### Import pysammos

In [1]:
import os
os.environ["NUMBA_NUM_THREADS"] = "8"
print(f">>> Numba is using {os.environ['NUMBA_NUM_THREADS']} cores")

>>> Numba is using 8 cores


In [1]:
import numpy as np

In [2]:
from pysammos.utils.config_loader import load_config
from pysammos.coarse_graining import CoarseGraining

Hello from pysammos
Loading macroscopic_fields package...
Loading data_read package...
Loading particle_phase package...
Loading spatial_weights package...
Loading neighbour_search package...
Loading grid_generation package...
Loading data_handle package...
Loading data_write package...


### Initialise the Coarse Graining class

In [3]:
# Load the configuration from the ini file
cfg = load_config("config.ini")  
print("-------------------- config.ini file read --------------------")
# Initialize the CoarseGraining class with the loaded configuration
CG = CoarseGraining(
    particle_path=cfg["particles_path"],
    contacts_path=cfg["contacts_path"],
    output_path=cfg["output_path"],
    start_timestep=cfg["t0"],
    end_timestep=cfg["tf"],
    dt_time_step=1,
    DEM_keymap=cfg["key_mapping"],
    grid_info=cfg["grid_info"],
    weight_function='Lucy',
    fields_to_export=cfg["fields_to_export"],
    ignore_phases=cfg["partialignore"]
                    ) 
print("  ") ; print("-------------------- CoarseGraining class initialised --------------------")

-------------------- config.ini file read --------------------
Output path already exists: ./PysammosCG/
  
-------------------- CoarseGraining class initialised --------------------


In [5]:
# 1. Load the size-relevant particle data for the first time step
Bounds_t0, Diameter_t0, Density_t0, Mass_t0, GlobalID_t0 = CG.data_sampling()

XML-based PolyData detected.


In [6]:
# 2. Calculate the particle size range
CG.get_particle_size_statistics(Diameter_t0, Mass_t0)
print(">> Particle size statistics: ") 
print("       d43: ", CG.d43)
print("       dmax: ", CG.dmax)
print("       d50: ", CG.d50)
print("       d32: ", CG.d32)
print("       drms: ", CG.drms)

>> Particle size statistics: 
       d43:  0.01651367411185649
       dmax:  0.019993
       d50:  0.016306963
       d32:  0.016109877148148044
       drms:  0.015422490500786803


In [10]:
# 3. Get the phases
CG.get_particle_phases(Diameter_t0, Density_t0, GlobalID_t0, 8)
print(">> Phases: ")
print("       Diameters: ", CG.phases[:,0])
print("       Densities: ", CG.phases[:,1])

Ignoring phases. Using mean density and d50 as phase.
>> Phases: 
       Diameters:  [0.01630696]
       Densities:  [2700.]


In [7]:
# 4. Calculate the CG grid spacing
CG.set_resolution(CG.d43) # here you can input different number, to make w and c bigger or smaller 
print(">> Coarse Graining resolution: ")
print("       c:", CG.c)
print("       w:", CG.w)

>> Coarse Graining resolution: 
       c: 0.024770511167784733
       w: 0.012385255583892366


In [8]:
# 5. Generate the CG grid
CG.generate_grid()
print(">> Grid: ")
print("       Grid Points: ", CG.GridPoints.shape, "First Point: ", CG.GridPoints[0])
print("       Nodes: ", CG.Nodes)
print("       Spacing: ", CG.Spacing)

Generating Grid with Customised Grid Ranges
Customised grid bounds: x = [0.00105, 0.5], y = [0.001, 0.24], z = [0.0, 0.02], x_transect = None, y_transect = None, z_transect = None
>> Grid: 
       Grid Points:  (2646, 3) First Point:  [0.00105 0.001   0.     ]
       Nodes:  [42 21  3]
       Spacing:  [0.01216951 0.01195    0.01      ]


In [11]:
# 6. Calculate the CG fields
CG.fields_in_time()

 
-------------------- Calculating Coarse Grained Fields --------------------
 
---> Time step 0: time 0150 ................................................
Loading data ... 
  Particle data loaded
  Repeated pairs in contact data:  1142
  Contact data loaded and mapped
  Coordination number not provided. Calculating it.
... data loaded
Matching particles to grid points ...
... particles assigned to grid nodes
Computing weights ...
... weights computed
Computing Coarse Graining fields...
  volume fraction done
  mixture density done
  momentum density done
shape of velocity gradient: (2646, 3, 3)
  contact tensor done
  particle density done
  total stress done
  d43 done
  d32 done
  Z done
  pressure done
  granular temp done
  shear rate done
  inertial number done
  mu done
  frabric tensor done
... fields computed
Writing results for timestep 150...
  File successfully updated to ./PysammosCG/CG_Lucy_Monodisperse.h5
  File successfully written to ./PysammosCG/CG_Lucy_Monodisperse_

### Example of sweep of w/d values

In [ ]:
# Load the configuration from the ini file
cfg = load_config("config.ini")  
print("-------------------- config.ini file read --------------------")
# Initialize the CoarseGraining class with the loaded configuration
CG = CoarseGraining(
    particle_path=cfg["particles_path"],
    contacts_path=cfg["contacts_path"],
    output_path=cfg["output_path"],
    start_timestep=cfg["t0"],
    end_timestep=cfg["tf"],
    dt_time_step=1,
    DEM_keymap=cfg["key_mapping"],
    grid_info=cfg["grid_info"],
    weight_function='Lucy',
    fields_to_export=cfg["fields_to_export"],
    ignore_phases=cfg["partialignore"]
                    ) 
print("  ") ; print("-------------------- CoarseGraining class initialised --------------------")

In [ ]:
CG.sweep_CG_widths(w_d=np.array([0.5, 0.8, 1.1]))
print(">> Sweep of CG widths completed.")
print(CG.GridPoints)